# Cross-Model Alignment: Structural Universality

This notebook demonstrates IRTK's `cross_model_alignment` module:

1. **CKA layer correspondence** – CKA similarity between models
2. **Match heads** – find corresponding heads across models
3. **Circuit universality** – test circuit universality across models
4. **Feature comparison** – align SAE features across models
5. **Scale law trajectory** – track metrics across model scales

## Why This Matters

Cross-model alignment measures whether different architectures learn corresponding representations. CKA correspondence, head matching, and circuit universality tests address a core question: do our interpretability findings generalize across models, or are they artifacts of specific architectures?

**Key references:**
- [Kornblith et al. (2019) "Similarity of Neural Network Representations"](https://arxiv.org/abs/1905.00414) — CKA for comparing representations across models and layers
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax, jax.numpy as jnp, numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.sae import SparseAutoencoder
from irtk.cross_model_alignment import (
    cka_layer_correspondence, match_heads_across_models,
    circuit_universality_score, aligned_feature_comparison,
    scale_law_trajectory,
)

In [ ]:
def make_model(seed):
    cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)
    model = HookedTransformer(cfg)
    key = jax.random.PRNGKey(seed)
    leaves, treedef = jax.tree.flatten(model)
    new_leaves = [jax.random.normal((key := jax.random.split(key)[1]), l.shape) * 0.1
                  if isinstance(l, jnp.ndarray) and l.dtype == jnp.float32 else l for l in leaves]
    return jax.tree.unflatten(treedef, new_leaves)

model_a, model_b = make_model(42), make_model(99)
tokens = jnp.array([0, 1, 2, 3, 4, 5])
def metric_fn(logits): return float(logits[-1, 0])

In [ ]:
cka = cka_layer_correspondence(model_a, model_b, tokens)
print(f"CKA matrix:\n{cka['cka_matrix'].round(4)}")
print(f"Best match: {cka['best_match']}")
print(f"Mean CKA: {cka['mean_cka']:.4f}")

In [ ]:
heads = match_heads_across_models(model_a, model_b, tokens)
print(f"Top 3 matches:")
for m in heads['matches'][:3]:
    print(f"  {m[0]} -> {m[1]}, score={m[2]:.4f}")

In [ ]:
models = [make_model(i) for i in range(3)]
univ = circuit_universality_score([(0,0), (1,0)], models, tokens, metric_fn)
print(f"Universality: {univ['universality_score']:.4f}")
print(f"Per-model faithfulness: {[round(f, 4) for f in univ['per_model_faithfulness']]}")

In [ ]:
sae_a = SparseAutoencoder(16, 32, key=jax.random.PRNGKey(0))
sae_b = SparseAutoencoder(16, 32, key=jax.random.PRNGKey(1))
feat = aligned_feature_comparison(sae_a, sae_b, model_a, model_b, tokens,
                                   'blocks.0.hook_resid_post', 'blocks.0.hook_resid_post')
print(f"Top matched pairs: {feat['matched_pairs'][:3]}")
print(f"Mean similarity: {feat['mean_similarity']:.4f}")

In [ ]:
trajectory = scale_law_trajectory(models, [100, 200, 400], tokens, metric_fn)
print(f"Sizes: {trajectory['sizes']}")
print(f"Metrics: {[round(m, 4) for m in trajectory['metrics']]}")
print(f"Trend: {trajectory['trend']}")
print(f"Scaling exponent: {trajectory['log_log_slope']:.4f}")